# 08 — TCN_starNLL: probabilistic Student-t extension

TCN_starNLL preserves TCN_star's inputs and causal backbone. It replaces the deterministic six-value head with location, positive scale, and bounded degrees-of-freedom outputs at each horizon and trains by Student-t negative log likelihood.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'outputs' / '06_tcn_development').is_dir())
OUT = ROOT / 'outputs' / '06_tcn_development'
runs = pd.read_csv(OUT / 'development_run_summary.csv')
horizons = pd.read_csv(OUT / 'development_horizon_metrics.csv')
architecture = pd.read_csv(OUT / 'architecture_comparison.csv')
training = pd.read_csv(OUT / 'training_contract_comparison.csv')


## Distribution contract

For every origin and horizon, `mu` is both the Student-t location and the reported point forecast in standardized target space. `scale = softplus(raw_scale) + 1e-4`. Degrees of freedom are bounded strictly between 2.1 and 20, initialized near 4, so variance remains finite while heavy tails remain available. All three heads are dynamic; this is not a fixed residual distribution attached after deterministic training.

In [ ]:
display(architecture.loc[architecture.model.isin(['TCN_star','TCN_starNLL'])].T)
display(training.loc[training.model.isin(['TCN_star','TCN_starNLL'])].T)

## Development evidence and comparison boundary

The deterministic model is selected by standardized MSE; the probabilistic model is selected by NLL. Those objective values are not placed on a common ranking scale. Point MAE/RMSE remain comparable because both models report the Student-t location or direct output as the point forecast. Calibration is evaluated later for the frozen ensemble.

In [ ]:
display(runs.loc[runs.model.isin(['TCN_star','TCN_starNLL'])].round(4))
display(horizons.loc[horizons.model.isin(['TCN_star','TCN_starNLL'])].round(4))
display(Image(filename=str(OUT / 'development_horizon_mae.png')))

## What is carried to the frozen evaluation

The public development evidence shows that probabilistic training preserves essentially the same point-forecast accuracy as TCN_star while defining case-specific predictive distributions. The official final comparison therefore evaluates both families as predeclared three-seed ensembles and reports NLL/calibration only where a predictive distribution genuinely exists.